<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/04_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Transfer Learning

ResNet18 с ImageNet-весами как фиксированный извлекатель признаков: замораживание backbone, замена `fc`, обучение head и сохранение лучшего checkpoint.

> Этот notebook самодостаточен: его можно запускать сверху вниз в чистом Google Colab. Он не требует выполнения других notebook-файлов проекта.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

raw_dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

labels = np.array([raw_dataset[i][1] for i in range(len(raw_dataset))])

train_indices, val_indices = train_test_split(
    np.arange(len(raw_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

class_names = raw_dataset.classes
weights = ResNet18_Weights.IMAGENET1K_V1
resnet_transform = weights.transforms()

resnet_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_train_dataset = Subset(resnet_train_source, train_indices.tolist())
resnet_val_dataset = Subset(resnet_val_source, val_indices.tolist())

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

resnet_val_loader = DataLoader(
    resnet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

loss_fn = nn.CrossEntropyLoss()

print("Train:", len(resnet_train_dataset))
print("Validation:", len(resnet_val_dataset))
print("Classes:", len(class_names))


In [ ]:
import torch
from sklearn.metrics import f1_score


def evaluate_metrics(model, dataloader, loss_fn, device, num_classes):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_top3_correct = 0
    total_objects = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3 = logits.topk(k=3, dim=1).indices

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_top3_correct += (
                (top3 == labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )
            total_objects += batch_size

            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )

    return {
        "loss": total_loss / total_objects,
        "accuracy": total_correct / total_objects,
        "macro_f1": macro_f1,
        "top3_accuracy": total_top3_correct / total_objects,
    }

## Transfer learning - перенос обучения

Наша модель `PetCNN` училась извлекать признаки с нуля на 2944 изображениях. Теперь возьмем **ResNet18 с весами, уже обученными на ImageNet**, и используем ее признаки для распознавания наших 37 пород


**Как будет устроена модель**

| **Часть** | **Что делает** | **Обучаем сейчас?** |
| :--- | :--- | :--- |
| Основная часть ResNet18 | Преобразует изображение в вектор из 512 признаков | Нет |
| Новый `Linear(512, 37)` | Преобразует признаки в 37 logits | Да |


Признаки основной части уже выучены на другом наборе изображений. Последний слой создадим со случайными весами - ему еще предстоит научиться связывать эти признаки с нашими породами

## Загрузка и подготовка модели

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

seed_everything(42)

weights = ResNet18_Weights.IMAGENET1K_V1
resnet_model = resnet18(weights=weights)

print("Исходный классификатор:", resnet_model.fc)

# Замораживаем параметры существующих слоев
for parameter in resnet_model.parameters():
  parameter.requires_grad = False

# Заменяем классификатор новый обучаемым слоем
in_features = resnet_model.fc.in_features

resnet_model.fc = nn.Linear(
    in_features,
    len(class_names)
)

resnet_model = resnet_model.to(device)

print("Новый классификатор:", resnet_model.fc)

trainable_parameters = sum(
    parameter.numel()
    for parameter in resnet_model.parameters()
    if parameter.requires_grad
)

print("Обучаемых параметров:", trainable_parameters)

Здесь важен порядок: **сначала замораживаем существующие параметры, затем создаем новый слой**. У нового `Linear` параметры по умолчанию имеют `requires_grad=True`


**Для предобученных весов нужна соответствующая обработка изображений**


In [ ]:
resnet_transform = weights.transforms()
print(resnet_transform)

Она уменьшает изображение с сохранением пропорций до короткой стороны 256, вырезает центральную область `224 × 224`, переводит значения в `[0, 1]` и нормализует каждый RGB-канал. Это обработка, рекомендованная для выбранных весов

Нормализация выглядит так:

$$ x_{normilized} = \frac{x - mean}{std} $$

После нее значения могут быть отрицательными и превышать единицу - это нормально

### Проверим одно изображение

Берем исходное PIL-изображение из `dataset`, где еще нет преобразований:

In [ ]:
image, label = raw_dataset[int(val_indices[0])]

image_tensor = resnet_transform(image)

# Добавляем размерность батча
inputs = image_tensor.unsqueeze(0).to(device)

resnet_model.eval()

with torch.inference_mode():
  logits = resnet_model(inputs)

print("Input shape:", inputs.shape)
print("Logits shape:", logits.shape)

## Обучение последнего слоя ResNet18

Теперь обучим **наш последний слой `ResNet18` на наших 37 породах**. Новая тема здесь: *как сохранить основную часть модели полностью замороженной*

**Почему одного `requires_grad=False` недостаточно?**

В ResNet18 есть слои `BatchNorm2d`. Они нормализуют промежуточные признаки и хранят накопленные средние значения и дисперсии.

В режиме `train` эти статистики обновляются при прохождении батчей - даже если параметры слоя заморожены. В режиме `eval` используются сохраненные статистики.

Поэтому для нашего эксперимента:
```
model.eval()        # Сохраняем статистики основной части
model.fc.train()    # Последний слой переводим в режим обучения
```

**`eval()` не отключает градиенты**. Последний слой может обучаться через `loss.backward()` и `optimizer.step()`. А вот оборачивать **весь** обучающий проход в `torch.inference_model()` нельзя.

У самого `Linear()` поведение в `train()` и `eval()` одинаковое, но такой записью мы явно обозначаем, какую часть обучаем.


## Функция обучения последнего слоя

Это уже знакомый цикл. Основное изменение - первые две строки внутри функции

In [ ]:
def train_head_one_epoch(
    model, dataloader, loss_fn, optimizer, device
):
  model.eval()
  model.fc.train()

  total_loss = 0
  total_correct = 0
  total_objects = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)
    total_loss += loss.item() * batch_size
    total_correct += (
        logits.argmax(dim=1) == labels
    ).sum().item()
    total_objects += batch_size

  return (
      total_loss / total_objects,
      total_correct / total_objects
  )

Здесь изображения проходят через всю сеть, но градиенты вычисляются только для нового `fc`

### Первый эксперимент - пять эпох

In [ ]:
seed_everything(42)
resnet_train_loader.generator.manual_seed(42)

resnet_model = resnet18(weights=weights)

for parameter in resnet_model.parameters():
  parameter.requires_grad = False

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    len(class_names),
)

resnet_model = resnet_model.to(device)

# Передаем оптимизатору только параметры последнего слоя
resnet_optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=0.001,
)
loss_fn = nn.CrossEntropyLoss()

resnet_checkpoint_path = Path(
    "checkpoints/resnet18_head.pth"
)

resnet_checkpoint_path.parent.mkdir(
    parents=True, exist_ok=True
)

resnet_history = []

best_macro_f1 = -float("inf")
best_epoch = None
epochs_without_improvement = 0

max_epochs = 15
patience = 3
min_delta = 1e-4

for epoch in range(1, max_epochs+1):
  train_loss, train_accuracy = train_head_one_epoch(
      resnet_model,
      resnet_train_loader,
      loss_fn,
      resnet_optimizer,
      device,
  )

  metrics = evaluate_metrics(
      resnet_model,
      resnet_val_loader,
      loss_fn,
      device,
      num_classes=len(class_names),
  )

  resnet_history.append({
      'epoch': epoch,
      'train_loss': train_loss,
      'train_accuracy': train_accuracy,
      'val_loss': metrics["loss"],
      'val_accuracy': metrics['accuracy'],
      'val_macro_f1': metrics['macro_f1'],
      'val_top3_accuracy': metrics['top3_accuracy'],
  })

  status = ""

  if metrics['macro_f1'] > best_macro_f1 + min_delta:
    best_macro_f1 = metrics['macro_f1']
    best_epoch = epoch
    epochs_without_improvement = 0

    torch.save({
        'model_state_dict': resnet_model.state_dict(),
        'epoch': epoch,
        'metrics': metrics,
        'class_names': class_names,
        'weights': 'IMAGENET1K_V1',
    }, resnet_checkpoint_path)

    status = 'saved'

  else:
    epochs_without_improvement += 1
    status = (
        f"No improvement: "
        f"{epochs_without_improvement}/{patience}"
    )

  print(
      f"Epoch {epoch}/{max_epochs} | "
      f"Train loss: {train_loss:.4f} | "
      f"Train accuracy: {train_accuracy:.2%} | "
      f"Val loss: {metrics['loss']:.4f} | "
      f"Val accuracy: {metrics['accuracy']:.2%} | "
      f"Val macro-F1: {metrics['macro_f1']:.4f} | "
      f"{status}"
  )

  if epochs_without_improvement >= patience:
    print("Early stopping")
    break

### Вернем лучшие веса

In [ ]:
checkpoint = torch.load(
    resnet_checkpoint_path,
    map_location=device,
    weights_only=True
)

resnet_model.load_state_dict(
    checkpoint['model_state_dict']
)

resnet_metrics = evaluate_metrics(
    resnet_model,
    resnet_val_loader,
    loss_fn,
    device,
    num_classes=len(class_names)
)

print("Loaded epoch:", checkpoint['epoch'])

for name, value in resnet_metrics.items():
  print(f"{name}: {value:.4f}")